# TP Final Jour 3 — Du dataset au modèle déployé

> Par BOUNYAMINE OUSMANOU

---
## PARTIE A — Chargement & choix des variables

### Étape 1 — Charger le fichier Excel et le convertir en CSV

`utf-8-sig` aux deux bouts : le CSV intermédiaire conserve les accents lisibles sous Excel, et se relit sans BOM parasite en tête de la première colonne.

In [19]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')

df = pd.read_excel('../data/dataset_assurance_ML.xlsx')
df.to_csv('../data/dataset_assurance_ML.csv', index=False, encoding='utf-8')

df = pd.read_csv('../data/dataset_assurance_ML.csv', encoding='utf-8')
print(df.shape)

(500, 27)


**(500, 27)**, aucun trou, aucun doublon : le fichier est celui nettoyé au Jour 2, on peut modéliser directement.

### Étape 2 — La variable cible

**Le problème est-il équilibré ? Quelle conséquence pour l'entraînement ?**

In [20]:
TARGET = 'Résiliation'
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(2))

Résiliation
0    450
1     50
Name: count, dtype: int64
Résiliation
0    0.9
1    0.1
Name: proportion, dtype: float64


> **Réponse:** **Non, les classes sont déséquilibrées : 450 clients restent (90 %) contre 50 qui résilient (10 %).**
>
> Trois conséquences concrètes :
> 1. **L'accuracy devient trompeuse** : un modèle qui répond toujours « reste » obtient 90 % d'accuracy sans rien apprendre. On jugera au **ROC-AUC**, au **recall** et au **F1** sur la classe minoritaire.
> 2. **`class_weight='balanced'`** : le modèle pondère les 50 résiliations pour qu'elles pèsent autant que les 450 autres, sinon il apprend simplement à toujours dire « reste ».
> 3. **Split stratifié obligatoire** : avec 50 positifs seulement, un tirage aléatoire pourrait déséquilibrer train et test et fausser l'évaluation.

### Étape 3 — Détecter une fuite de données

In [21]:
print(pd.crosstab(df['Statut Contrat'], df[TARGET], margins=True))

Résiliation       0   1  All
Statut Contrat              
Actif           450   0  450
Résilié           0  36   36
Suspendu          0  14   14
All             450  50  500


> **Réponse.** **`Statut Contrat` révèle la cible à 100 %** : `Actif` → 450 clients tous à 0, `Résilié` → 36 tous à 1, `Suspendu` → 14 tous à 1. Aucune case ne se mélange.
>
> **Non, on ne peut pas l'utiliser.** Un modèle qui la verrait afficherait un score parfait en test… et serait **inutilisable en production** : au moment où le conseiller veut anticiper un départ, le statut vaut encore « Actif » pour tout le monde. L'information n'existe qu'**après** le fait qu'on cherche à prédire.
>
> **Le réflexe à garder** : pour chaque variable, se demander « est-ce que je connaîtrais cette valeur **avant** que le client ne résilie ? ». Si non, elle sort. Même logique pour `Date Échéance` (la date de sortie) et les identifiants.

### Étape 4 — Choisir les variables numériques et catégorielles

12 variables : les plus liées à la cible d'après l'EDA du Jour 2, et un nombre de champs qui reste saisissable par un conseiller au téléphone.

**Variables écartées volontairement :**

| Exclue | Raison |
|---|---|
| `Statut Contrat` | Fuite de données (étape 3) |
| `N° Police`, `Nom`, `Prénom` | Identifiants — aucun pouvoir prédictif, et données personnelles |
| `Code Postal`, `Ville` | Géographie peu liée à la cible, et 10 modalités de plus à saisir |
| `Date Souscription`, `Date Échéance` | Dates brutes ; l'information utile est déjà dans `Ancienneté (mois)` |
| `Sexe` | Variable protégée : la tarification et le ciblage sur le sexe sont interdits en assurance dans l'UE (arrêt Test-Achats, 2012) |
| `Franchise`, `Nb Garanties`, `Type/Puissance/Valeur Véhicule` | Signal faible et forte colinéarité avec la prime — 5 champs de saisie pour peu de gain |

In [22]:
num_cols = ['Âge', 'Salaire Annuel (€)', 'Prime Annuelle (€)', 'Ancienneté (mois)',
            'Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)',
            'Montant Sinistres (€)', 'Score Risque (0-100)']

cat_cols = ['Type Contrat', 'Catégorie Prof.', 'Usage Véhicule', 'Dernier Sinistre']

# Garde-fou : les noms contiennent accents, espaces et symboles — une lettre suffit à faire échouer
manquantes = [c for c in num_cols + cat_cols if c not in df.columns]
assert not manquantes, f'Colonnes introuvables : {manquantes}'

X = df[num_cols + cat_cols]
y = df[TARGET]
print(X.shape, y.shape)

(500, 12) (500,)


### Étape 5 — Vérifier le lien avec la cible

In [23]:
print(X[num_cols].corrwith(y).round(3).sort_values(ascending=False))
print()
print(df.groupby('Dernier Sinistre')[TARGET].mean().round(2).sort_values())

Nb Sinistres (3 ans)     0.455
Score Risque (0-100)     0.441
Coeff. Bonus-Malus       0.412
Montant Sinistres (€)    0.284
Prime Annuelle (€)       0.041
Âge                     -0.006
Salaire Annuel (€)      -0.012
Ancienneté (mois)       -0.055
dtype: float64

Dernier Sinistre
Aucun                    0.03
Incendie                 0.16
Catastrophe naturelle    0.18
Accident                 0.26
Bris de glace            0.30
Vol                      0.30
Dégât des eaux           0.36
Name: Résiliation, dtype: float64


In [24]:
for c in cat_cols:
    print(df.groupby(c)[TARGET].mean().round(3).sort_values(ascending=False).to_dict())

{'Platine': 0.13, 'Gold': 0.114, 'Bronze': 0.102, 'Silver': 0.082}
{'Entrepreneur': 0.2, 'Retraité': 0.118, 'Ouvrier': 0.109, 'Cadre': 0.104, 'Technicien': 0.104, 'Fonctionnaire': 0.093, 'Employé': 0.079, 'Profession libérale': 0.05}
{'Professionnel': 0.118, 'Domicile-Travail': 0.1, 'Loisirs': 0.098, 'Mixte': 0.084}
{'Dégât des eaux': 0.36, 'Bris de glace': 0.3, 'Vol': 0.296, 'Accident': 0.261, 'Catastrophe naturelle': 0.176, 'Incendie': 0.158, 'Aucun': 0.026}


> **Réponse — les 3 variables numériques les plus liées à la résiliation :**
> 1. **`Nb Sinistres (3 ans)` : +0,455**
> 2. **`Score Risque (0-100)` : +0,441**
> 3. **`Coeff. Bonus-Malus` : +0,412**
>
> Puis `Montant Sinistres` (+0,284). Les quatre racontent la même histoire : **la sinistralité est le moteur de la résiliation**. Un client qui déclare des sinistres voit son bonus-malus se dégrader, son score de risque monter, sa prime augmenter au renouvellement — et il part.
>
> À l'inverse, `Prime Annuelle` (+0,041), `Âge` (−0,006), `Salaire` (−0,012) et `Ancienneté` (−0,055) sont **quasiment sans lien**. Le prix absolu ne fait pas partir ; **c'est la hausse consécutive aux sinistres qui fait partir.**
>
> `Dernier Sinistre` confirme : **3 %** de résiliation chez ceux qui n'en ont aucun, contre **26 à 36 %** dès qu'un sinistre est survenu (Dégât des eaux 36 %, Bris de glace et Vol 30 %, Accident 26 %). C'est un facteur **10**, le signal le plus net du dataset.

⚠️ **Écart avec l'énoncé.** Le PDF annonce Score Risque 0,245 / Nb Sinistres 0,237 / Bonus-Malus 0,232, et « Vol 37 % · Bris de glace 27 % ». Le fichier `dataset_assurance_ML.xlsx` fourni donne des corrélations **nettement plus fortes** (0,455 / 0,441 / 0,412) et un ordre légèrement différent. Les chiffres ci-dessus sont ceux réellement calculés sur le fichier — le classement des trois premières variables et la conclusion métier sont inchangés. Même remarque plus bas pour l'AUC.

---
## PARTIE B — Pipeline, entraînement & évaluation

### Étape 6 — Séparer train / test

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(round(y_train.mean(), 2), round(y_test.mean(), 2))

(400, 12) (100, 12)
0.1 0.1


### Étape 7 — Le prétraitement en un seul objet : ColumnTransformer

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

`handle_unknown='ignore'` est un choix de **robustesse en production** : si l'interface envoie un jour une modalité jamais vue à l'entraînement, l'encodeur produit une ligne de zéros au lieu de faire planter l'application.

### Étape 8 — Deux candidats dans un Pipeline

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

candidats = {
    'Régression Logistique': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=10,
        class_weight='balanced', random_state=42),
}

pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)])
             for nom, algo in candidats.items()}
print(list(pipelines))

['Régression Logistique', 'Random Forest']


`max_depth=4` et `min_samples_leaf=10` bridés volontairement : avec **400 lignes d'entraînement dont 40 positifs**, une forêt sans limite mémoriserait le train et s'effondrerait en test. Contraindre la profondeur est ici plus efficace que d'ajouter des arbres.

### Étape 9 — Comparer par validation croisée

In [28]:
from sklearn.model_selection import cross_val_score

for nom, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}')

Régression Logistique  AUC = 0.745 ± 0.113
Random Forest          AUC = 0.801 ± 0.096


> **Réponse — lequel retenir ? L'écart est-il significatif ?**
>
> Régression Logistique **0,745 ± 0,113** · Random Forest **0,801 ± 0,096**.
>
> L'écart de 0,056 en faveur du Random Forest est **réel mais pas décisif** : il reste inférieur à l'écart-type entre les 5 plis (≈ 0,10), donc sur un autre découpage les deux pourraient se croiser. Avec 400 lignes, cette incertitude est normale.
>
> **On retient le Random Forest** pour trois raisons : la moyenne est meilleure, la variance légèrement plus faible, et surtout il expose `feature_importances_` — indispensable pour l'onglet « variables influentes » de l'interface. Le modèle le plus utile n'est pas seulement le plus précis : c'est aussi celui qu'on peut expliquer à un conseiller.

*Note : l'énoncé annonçait ≈ 0,62 pour les deux. Sur le fichier fourni, le signal est plus fort et l'écart entre les deux modèles plus marqué.*

### Étape 10 — Entraîner le modèle retenu et évaluer sur le test

In [40]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)

pipeline = pipelines['Random Forest']
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print('Accuracy :', round(accuracy_score(y_test, y_pred), 3))
print('F1       :', round(f1_score(y_test, y_pred), 3))
print('ROC-AUC  :', round(roc_auc_score(y_test, y_proba), 3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie']))

Accuracy : 0.87
F1       : 0.519
ROC-AUC  : 0.853
[[80 10]
 [ 3  7]]
              precision    recall  f1-score   support

       Reste       0.96      0.89      0.92        90
     Résilie       0.41      0.70      0.52        10

    accuracy                           0.87       100
   macro avg       0.69      0.79      0.72       100
weighted avg       0.91      0.87      0.88       100



### Étape 11 — Lire la matrice de confusion comme un métier

Résultats obtenus : **Accuracy 0,870 · F1 0,519 · ROC-AUC 0,853**, matrice `[[80 10] [3 7]]`.
Autrement dit : sur 10 résiliations réelles, le modèle en détecte **7** (recall 70 %), au prix de **10 fausses alertes** (précision 41 %).

#### Q1. Un modèle qui prédirait toujours « reste » aurait 90 % d'accuracy. Est-il meilleur ?

> **Non — il est strictement inutile.** Il aurait 90 % d'accuracy contre 87 % pour le nôtre, et pourtant il détecterait **zéro** résiliation : recall 0 %, F1 0, AUC 0,5 (le hasard). Le service Fidélisation n'aurait aucun client à appeler.
>
> C'est exactement le piège de l'accuracy sur classes déséquilibrées : notre modèle « perd » 3 points d'accuracy parce qu'il **ose** signaler des clients, et c'est précisément ce qu'on lui demande. Avec un AUC de **0,853**, il ordonne correctement les clients par risque dans 85 % des paires — c'est cela qui a de la valeur.

#### Q2. Quelle erreur coûte le plus cher : faux négatif ou faux positif ?

> **Le faux négatif, sans comparaison possible.**
>
> - **Faux positif** (10 cas) : on appelle un client qui serait resté. Coût = quelques minutes de conseiller, éventuellement un geste commercial. Ordre de grandeur : **quelques dizaines d'euros**, et l'appel peut même renforcer la relation.
> - **Faux négatif** (3 cas) : un client part sans qu'on ait rien tenté. Coût = la prime annuelle perdue (≈ 890 € ici), la marge sur toutes les années suivantes, et le coût d'acquisition d'un remplaçant — en assurance, recruter un client coûte couramment **5 à 10 fois** plus cher que d'en retenir un.
>
> Le rapport de coût est de l'ordre de **1 pour 20**, voire davantage. Rater un départ est l'erreur à éviter en priorité.

#### Q3. Faut-il baisser ou monter le seuil de 0,5 ?

> **Le baisser — c'est la règle générale, mais le tableau ci-dessous nuance le cas présent.**
>
> Le principe d'abord : baisser le seuil rend le modèle plus alarmiste, il signale plus de clients, donc **rate moins de départs** (recall ↑) au prix de **plus de fausses alertes** (précision ↓). Vu le rapport de coût du Q2, c'est le bon sens de l'arbitrage.
>
> **Ce que montrent les chiffres sur ce test :** le recall **plafonne à 70 %** dès 0,55 et ne bouge plus en descendant. Les 3 résiliations manquées ont des probabilités très basses — le modèle ne les voit pas du tout, et descendre le seuil n'ajoute que des fausses alertes (à 0,30 : 27 alertes pour les mêmes 7 détections, précision effondrée à 26 %). Monter à 0,70 fait en revanche perdre 2 détections sur 7.
>
> **Le meilleur compromis ici est donc ≈ 0,55** : 12 alertes, 7 départs détectés, 58 % de précision. Attention toutefois : avec 10 positifs seulement dans le test, ce tableau est instable — à confirmer sur un échantillon plus large.
>
> Et le bon seuil reste avant tout une question **capacitaire** : combien d'appels par semaine l'équipe peut-elle passer ? On trie le portefeuille par probabilité décroissante et on descend jusqu'à saturer cette capacité. C'est pourquoi l'interface (partie D) expose un **curseur « Seuil d'alerte »** de 0,30 à 0,70 : ce réglage appartient au métier, pas au modèle.

In [31]:
# Effet du seuil sur le compromis recall / précision
from sklearn.metrics import precision_score, recall_score

print(f"{'Seuil':>6} {'Alertes':>8} {'Détectés':>9} {'Recall':>8} {'Précision':>10}")
for seuil in [0.30, 0.40, 0.50, 0.55, 0.60, 0.70]:
    pred = (y_proba >= seuil).astype(int)
    print(f'{seuil:>6.2f} {pred.sum():>8d} {int(((pred == 1) & (y_test == 1)).sum()):>9d} '
          f'{recall_score(y_test, pred, zero_division=0):>8.2f} '
          f'{precision_score(y_test, pred, zero_division=0):>10.2f}')

 Seuil  Alertes  Détectés   Recall  Précision
  0.30       27         7     0.70       0.26
  0.40       24         7     0.70       0.29
  0.50       17         7     0.70       0.41
  0.55       12         7     0.70       0.58
  0.60       12         7     0.70       0.58
  0.70        6         5     0.50       0.83


---
## PARTIE C — Sauvegarde & interrogation du modèle

### Étape 12 — Sauvegarder le pipeline complet

In [32]:
import joblib, os

chemin_pkl = MODELS / 'pipeline_resiliation.pkl'
joblib.dump(pipeline, chemin_pkl)
print(round(os.path.getsize(chemin_pkl) / 1024, 1), 'Ko')

493.7 Ko


Le `.pkl` contient **le scaler fitté, l'encodeur fitté et la forêt** : un seul fichier à déployer, et aucun risque d'oublier une transformation côté application.

### Étape 13 — Sauvegarder les métadonnées pour l'interface

In [33]:
import json

meta = {
    'modele': 'Random Forest',
    'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3),
    'f1_test': round(float(f1_score(y_test, y_pred)), 3),
    'taux_resiliation': round(float(y.mean()), 3),
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'num_ranges': {c: {'min': float(X[c].min()), 'max': float(X[c].max()),
                       'median': float(X[c].median())} for c in num_cols},
    'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols},
    # Bonus : distribution des probabilités du portefeuille, pour situer un client
    'proba_portefeuille': [round(float(p), 4) for p in pipeline.predict_proba(X)[:, 1]],
}

with open(MODELS / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('Âge :', meta['num_ranges']['Âge'])
print('Type Contrat :', meta['cat_values']['Type Contrat'])

Âge : {'min': 18.0, 'max': 79.0, 'median': 50.0}
Type Contrat : ['Bronze', 'Gold', 'Platine', 'Silver']


`ensure_ascii=False` est indispensable : sans lui, « Âge » serait écrit `\u00c2ge` dans le fichier — lisible par la machine, illisible pour vous quand il faudra déboguer.

**Pourquoi un fichier de métadonnées séparé ?** Parce que l'interface a besoin de savoir quoi afficher : bornes des curseurs, modalités des menus, noms exacts des colonnes. En les lisant ici plutôt qu'en les écrivant en dur dans `app.py`, un ré-entraînement sur de nouvelles données met l'interface à jour **sans toucher une ligne de code**.

### Étape 14 — Interroger le modèle sur un nouveau client

In [34]:
# Nouvelle cellule : on repart de zéro, comme le ferait l'application
modele = joblib.load(MODELS / 'pipeline_resiliation.pkl')

client = pd.DataFrame([{
    'Âge': 34, 'Salaire Annuel (€)': 28000, 'Prime Annuelle (€)': 950,
    'Ancienneté (mois)': 6, 'Coeff. Bonus-Malus': 1.25, 'Nb Sinistres (3 ans)': 3,
    'Montant Sinistres (€)': 4200, 'Score Risque (0-100)': 72,
    'Type Contrat': 'Bronze', 'Catégorie Prof.': 'Entrepreneur',
    'Usage Véhicule': 'Professionnel', 'Dernier Sinistre': 'Vol',
}])

print('Classe :', modele.predict(client))
print('Proba  :', round(float(modele.predict_proba(client)[0, 1]), 3))

Classe : [1]
Proba  : 0.816


In [35]:
# Le même client, mais avec un profil fidèle
fidele = client.copy()
fidele.loc[0, ['Ancienneté (mois)', 'Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)',
               'Montant Sinistres (€)', 'Score Risque (0-100)',
               'Type Contrat', 'Dernier Sinistre']] = [200, 0.60, 0, 0, 10, 'Gold', 'Aucun']

print('Classe :', modele.predict(fidele))
print('Proba  :', round(float(modele.predict_proba(fidele)[0, 1]), 3))

Classe : [0]
Proba  : 0.269


**Profil à risque → 82 %** (classe 1) contre **27 %** pour le profil fidèle : un écart de 55 points obtenu en ne changeant que la sinistralité et l'ancienneté.

Remarquez surtout ce qui vient d'être envoyé au modèle : du **texte** (`'Bronze'`, `'Vol'`) et des **euros bruts**. Aucun scaling, aucun encodage manuel — le pipeline a tout fait. C'est exactement ce qui rendra `app.py` court.

*(L'énoncé annonce ≈ 0,668 et 0,17 ; l'écart vient du fichier de données, cf. étape 5.)*

### Étape 15 — Provoquer l'erreur classique, puis passer en script

In [36]:
try:
    modele.predict(client.drop(columns=['Score Risque (0-100)']))
except Exception as e:
    print('ERREUR :', e)

ERREUR : columns are missing: {'Score Risque (0-100)'}


> **Réponse — quelle règle en tirer pour l'interface ?**
>
> `ColumnTransformer` sélectionne les colonnes **par leur nom exact**. Il en manque une : refus net, pas de valeur par défaut, pas de devinette.
>
> **Règle : l'interface doit fournir exactement les 12 colonnes, avec les noms exacts (accents, espaces, symbole €, parenthèses compris).** D'où trois choix dans `app.py` :
> 1. les noms viennent de `metadata.json`, jamais retapés à la main ;
> 2. le DataFrame est reconstruit dans l'ordre `num_cols + cat_cols` avant chaque prédiction ;
> 3. l'onglet « scoring par lot » vérifie les colonnes du CSV importé et affiche celles qui manquent, plutôt que de laisser remonter une trace d'erreur brute à l'utilisateur.
>
> C'est aussi une bonne nouvelle : cette rigidité **empêche de prédire sur des données mal formées** sans s'en apercevoir.

### Passage en script

Tout le code des parties A à C est regroupé dans **`train_model.py`** à la racine du projet. Une commande recrée le `.pkl`, le `.json` et `requirements.txt` :

```bash
python train_model.py
```

Un notebook s'exécute cellule par cellule, dans un ordre que seul son auteur connaît ; un script se relance en une ligne, sur n'importe quelle machine. C'est la condition d'un projet qu'on déploie — et c'est ce qui permettra de ré-entraîner le modèle sur de nouvelles données sans rouvrir ce notebook.